# Market Modeling and System Operation

This notebook adapts two of [PyPSA](https://pypsa.org)'s own official example notebooks to
Zambia:

1. Market Modeling — adapted from
   [`demand-supply-bids`](https://docs.pypsa.org/latest/examples/demand-supply-bids/),
   showing how a market operator clears competing generation and demand bids into a single
   price, first at a single node and then across two interconnected Zambian zones.
2. System Operation — adapted from
   [`scigrid-lopf-then-pf`](https://docs.pypsa.org/latest/examples/scigrid-lopf-then-pf/),
   showing how a system operator checks that an economically optimal dispatch is also
   *physically* feasible on the real network, using a non-linear power flow.


In [ ]:
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path
import pypsa

## 1. Market Modeling

This section reproduces PyPSA's
[`demand-supply-bids`](https://docs.pypsa.org/latest/examples/demand-supply-bids/) example,
with the two market zones renamed **Copperbelt** and **Lusaka**, Zambia's two main demand
centres.

In [ ]:
plt.style.use("bmh")
supply_bids = {
    "qty": [120, 100, 50, 60, 70, 60, 50],
    "price": [5, 15, 20, 36, 60, 150, 200],
}

demand_bids = {
    "qty": [250, 80, 20, 40, 60],
    "price": [200, 90, 75, 65, 24],
}

In [ ]:
def clearing_point(supply_qty, supply_price, demand_qty, demand_price):
    """Quantity and price where the merit-order supply and demand curves cross."""
    s_cum = np.cumsum(supply_qty)
    d_cum = np.cumsum(demand_qty)
    q_max = int(min(s_cum[-1], d_cum[-1]))
    price, qty = supply_price[0], 0
    for q in range(1, q_max + 1):
        s_p = supply_price[np.searchsorted(s_cum, q)]
        d_p = demand_price[np.searchsorted(d_cum, q)]
        if s_p > d_p:
            break
        price, qty = s_p, q
    return qty, price

### Single market zone

First, we treat the whole country as a single .

In [ ]:
qty, price = clearing_point(
    supply_bids["qty"], supply_bids["price"], demand_bids["qty"], demand_bids["price"]
)

plt.figure(figsize=(8, 4))
plt.step(
    np.cumsum([0] + supply_bids["qty"]),
    supply_bids["price"][:1] + supply_bids["price"],
    label="Supply",
)
plt.step(
    np.cumsum([0] + demand_bids["qty"]),
    demand_bids["price"][:1] + demand_bids["price"],
    label="Demand",
)
plt.scatter([qty], [price], color="k", zorder=5)
plt.annotate(
    f"Clearing price: ${price}/MWh\nat {qty} MW",
    xy=(qty, price),
    xytext=(qty + 20, price + 20),
    arrowprops={"arrowstyle": "->"},
)
plt.xlabel("Quantity (MW)")
plt.ylabel("Price (USD/MWh)")
plt.legend()

PyPSA

In [ ]:
n = pypsa.Network()
n.add("Bus", "Zambia")

# Add supply bids
n.madd(
    "Generator",
    names=[f"supply_{i}" for i in range(len(supply_bids["qty"]))],
    bus="Zambia",
    marginal_cost=supply_bids["price"],
    p_nom=supply_bids["qty"],
)

# Add demand bids
n.madd(
    "Generator",
    names=[f"demand_{i}" for i in range(len(demand_bids["qty"]))],
    bus="Zambia",
    p_nom=demand_bids["qty"],
    marginal_cost=[-p for p in demand_bids["price"]],
    sign=-1,
)

In [ ]:
n.optimize(solver_name="highs")

The market-clearing objective, and the nodal balance constraint the Market Operator has to satisfy:

In [ ]:
display(n.model.objective)
n.model.constraints["Bus-nodal_balance"]

In [ ]:
display(n.model.constraints["Generator-fix-p-upper"])
display(n.model.constraints["Generator-fix-p-lower"])

The single, uniform clearing price for the whole country, and the resulting dispatch:

In [ ]:
n.buses_t.marginal_price

In [ ]:
n.generators_t.p

### Two zones: Copperbelt and Lusaka

Now we split supply and demand between the **Copperbelt** and **Lusaka**

In [ ]:
supply_bids["zone"] = ["Copperbelt", "Lusaka", "Copperbelt", "Lusaka", "Lusaka", "Lusaka", "Lusaka"]
demand_bids["zone"] = ["Lusaka", "Copperbelt", "Copperbelt", "Lusaka", "Copperbelt"]

In [ ]:
plt.figure(figsize=(8, 4))

for zone, linestyle in zip(["Copperbelt", "Lusaka"], ["-", "--"]):
    s_idx = [i for i, z in enumerate(supply_bids["zone"]) if z == zone]
    d_idx = [i for i, z in enumerate(demand_bids["zone"]) if z == zone]

    s_qty = [supply_bids["qty"][i] for i in s_idx]
    s_price = [supply_bids["price"][i] for i in s_idx]
    d_qty = [demand_bids["qty"][i] for i in d_idx]
    d_price = [demand_bids["price"][i] for i in d_idx]

    plt.step(
        np.cumsum([0] + s_qty),
        [s_price[0]] + s_price,
        label=f"Supply ({zone})",
        color="C0",
        linestyle=linestyle,
    )
    plt.step(
        np.cumsum([0] + d_qty),
        [d_price[0]] + d_price,
        label=f"Demand ({zone})",
        color="C1",
        linestyle=linestyle,
    )

    qty, price = clearing_point(s_qty, s_price, d_qty, d_price)
    plt.scatter([qty], [price], color="k", zorder=5)
    plt.annotate(
        f"{zone}: ${price}/MWh",
        xy=(qty, price),
        xytext=(qty + 10, price + 15),
        arrowprops={"arrowstyle": "->"},
    )

plt.xlabel("Quantity (MW)")
plt.ylabel("Price (USD/MWh)")
plt.legend()
plt.tight_layout()

Without any interconnection between the two zones, each clears independently.

PyPSA

In [ ]:
n2 = pypsa.Network()

n2.add("Bus", "Copperbelt")
n2.add("Bus", "Lusaka")

n2.madd(
    "Generator",
    names=[f"supply_{i}" for i in range(len(supply_bids["qty"]))],
    bus=supply_bids["zone"],
    marginal_cost=supply_bids["price"],
    p_nom=supply_bids["qty"],
)
n2.madd(
    "Generator",
    names=[f"demand_{i}" for i in range(len(demand_bids["qty"]))],
    bus=demand_bids["zone"],
    p_nom=demand_bids["qty"],
    marginal_cost=[-p for p in demand_bids["price"]],
    sign=-1,
)

In [ ]:
n2.optimize(solver_name="highs")

In [ ]:
n2.buses_t.marginal_price

Splitting the market into two isolated zones destroys welfare relative to the single, national market:

In [ ]:
display(n.objective)
display(n2.objective)

### Adding the Copperbelt–Lusaka transmission link

With a transmission line between the two zones, so power (and price convergence) can actually flow between the Copperbelt and Lusaka.

In [ ]:
n2.add("Line", "Copperbelt-Lusaka", bus0="Copperbelt", bus1="Lusaka", s_nom=100)
n2.optimize(solver_name="highs")

In [ ]:
display(n2.objective)
display(n2.buses_t.marginal_price)

With enough transfer capacity on the interconnecting line, the two zonal prices converge
back towards the single-market outcome.

## 2. System Operation

Clearing a market is not enough on its own System Operator function exists because a
dispatch that is economically optimal is not automatically physically feasible on the real
grid. Lines have thermal limits, an N-1 security margin needs to be kept in reserve for
contingencies, and voltage/reactive power has to stay within bounds.
This section reproduces PyPSA's
[`scigrid-lopf-then-pf`](https://docs.pypsa.org/latest/examples/scigrid-lopf-then-pf/) example, optimise dispatch with a linear model, then re-check it with a full non-linear power flow,
but on the actual PyPSA-Zambia network.


In [ ]:

_candidates = [
    Path("../results/benchmark_run/networks/elec_s_all_ec_lv1.0_Co2L-3h.nc"),
    Path("results/benchmark_run/networks/elec_s_all_ec_lv1.0_Co2L-3h.nc"),
]
network_path = next((p for p in _candidates if p.exists()), _candidates[0])

n = pypsa.Network(str(network_path))
n

Load Distribution

In [ ]:
fig, ax = plt.subplots(
    1,
    1,
    subplot_kw={"projection": ccrs.EqualEarth()},
)

load_distribution = n.loads_t.p_set.iloc[0].groupby(n.loads.bus).sum()
n.plot(bus_sizes=load_distribution / 6000, ax=ax, title="Zambia — load distribution");

In [ ]:
n.generators.groupby("carrier")["p_nom"].sum().round(1)

In [ ]:
n.storage_units.groupby("carrier")["p_nom"].sum().round(1)

In [ ]:
techs = ["hydro", "ror", "coal", "oil", "biomass", "solar", "onwind"]

n_graphs = len(techs)
n_cols = 3
if n_graphs % n_cols == 0:
    n_rows = n_graphs // n_cols
else:
    n_rows = n_graphs // n_cols + 1


fig, axes = plt.subplots(
    nrows=n_rows, ncols=n_cols, subplot_kw={"projection": ccrs.EqualEarth()}
)
size = 6
fig.set_size_inches(size * n_cols, size * n_rows)
axes = axes.flatten()

for i, tech in enumerate(techs):
    ax = axes[i]
    if tech in n.storage_units.carrier.unique():
        units = n.storage_units[n.storage_units.carrier == tech]
    else:
        units = n.generators[n.generators.carrier == tech]
    tech_distribution = (
        units.groupby("bus")["p_nom"].sum().reindex(n.buses.index, fill_value=0)
    )
    n.plot(ax=ax, bus_sizes=tech_distribution / 6000)
    ax.set_title(tech)

for j in range(len(techs), len(axes)):
    fig.delaxes(axes[j])

fig.tight_layout()

### Dispatch optimisation with an N-1 security margin

In [ ]:
n.set_snapshots(n.snapshots[:8])  # first day at 3-hourly resolution

contingency_factor = 0.7
n.lines.s_max_pu = contingency_factor

In [ ]:
n.optimize(solver_name="highs")

## Plot dispatch time series

In [ ]:
p_by_carrier = n.generators_t.p.T.groupby(n.generators.carrier).sum().T
p_by_carrier = p_by_carrier.loc[:, (p_by_carrier != 0).any()]
p_by_carrier.columns

In [ ]:
colors = n.carriers.color.reindex(p_by_carrier.columns)

fig, ax = plt.subplots()
p_by_carrier.div(1e3).plot(kind="area", ax=ax, lw=0, color=colors, alpha=0.7)
ax.legend(ncol=3, loc="upper left", bbox_to_anchor=(0, 1.02, 1, 0.2), frameon=False)
ax.set_ylabel("GW")
ax.set_xlabel("")

## Plot hydro storage time series

Zambia's dispatchable hydro (Kariba, Kafue Gorge and similar) is modelled as a storage unit rather than a run-of-river generator, so its reservoir behaviour shows up here:

In [ ]:
fig, ax = plt.subplots()

p_storage = n.storage_units_t.p.sum(axis=1)
state_of_charge = n.storage_units_t.state_of_charge.sum(axis=1)
p_storage.plot(label="Hydro storage dispatch", ax=ax)
state_of_charge.plot(label="State of charge", ax=ax)

ax.axhline(0, color="k", lw=0.5, ls="--")
ax.legend()
ax.set_ylabel("MWh")
ax.set_xlabel("")

## Line loading from optimisation

With the linear power flow, there is the following per-unit loading at the fourth snapshot:

In [ ]:
now = n.snapshots[4]
loading = n.lines_t.p0.loc[now] / n.lines.s_nom
loading.describe()

In [ ]:
fig, ax = plt.subplots(subplot_kw={"projection": ccrs.EqualEarth()})
n.plot(
    ax=ax,
    line_colors=loading.abs(),
    line_cmap="viridis",
    title="Line loading",
    bus_sizes=1e-3,
);

## Locational marginal prices

Let's have a look at the distribution of marginal prices across ISMO's grid:

In [ ]:
n.buses_t.marginal_price.loc[now]

In [ ]:
fig, ax = plt.subplots(subplot_kw={"projection": ccrs.PlateCarree()})

plt.hexbin(
    n.buses.x,
    n.buses.y,
    gridsize=10,
    C=n.buses_t.marginal_price.loc[now],
    cmap="viridis",
    zorder=-1,
)
n.plot(ax=ax, line_widths=1, bus_sizes=0)

cb = plt.colorbar(location="right")
cb.set_label("Locational Marginal Price (USD/MWh)")

## Curtailment

By considering how much power is available and how much is generated, we can see what share
of variable renewables ISMO would need to curtail:

In [ ]:
carrier = "solar"

capacity = n.generators.groupby("carrier").sum().at[carrier, "p_nom"]
p_available = n.generators_t.p_max_pu.multiply(n.generators["p_nom"])
p_available_by_carrier = p_available.T.groupby(n.generators.carrier).sum().T
p_curtailed_by_carrier = p_available_by_carrier - p_by_carrier

In [ ]:
p_df = pd.DataFrame(
    {
        carrier + " available": p_available_by_carrier[carrier],
        carrier + " dispatched": p_by_carrier[carrier],
        carrier + " curtailed": p_curtailed_by_carrier[carrier],
    }
)

p_df[carrier + " capacity"] = capacity
p_df.loc[p_df[carrier + " curtailed"] < 0, carrier + " curtailed"] = 0

In [ ]:
fig, ax = plt.subplots()
p_df[[carrier + " dispatched", carrier + " curtailed"]].plot(kind="area", ax=ax, lw=0)
p_df[[carrier + " available", carrier + " capacity"]].plot(ax=ax)

ax.set_xlabel("")
ax.set_ylabel("Power [MW]")
ax.legend()

## Non-linear power flow


In [ ]:
n.optimize.fix_optimal_dispatch()

In [ ]:
n.generators.control.value_counts()

In [ ]:
info = n.pf()

Any failed to converge?

In [ ]:
(~info.converged).any().any()

Line flows

In [ ]:
n.lines_t.p0

voltage angle differences across the lines

In [ ]:
n.buses_t.v_ang * 180 / np.pi

Bus voltage

In [ ]:
n.buses_t.v_mag_pu